# Diario Federal — prueba aislada de VideoAgent (Kaggle)

Esta prueba ejecuta **un componente real de VideoAgent**: transcripción de un clip con Whisper y preguntas/resumen sobre la transcripción usando Gemini. No instala el conjunto completo de VideoAgent, no abre servidores web y no modifica Diario Federal.

**Antes de empezar:** en Kaggle, activar Internet y GPU (si hay disponibilidad); agregar `GEMINI_API_KEY` en Add-ons → Secrets; adjuntar un video breve (1–3 minutos) como entrada privada. No usar material confidencial ni subir videos a un dataset público.

El notebook usa una sesión temporal. No es un servicio permanente ni una integración productiva.

In [ ]:
import os, shutil, subprocess, sys, torch
print('Python:', sys.version)
print('GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Espacio libre (GB):', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))
print('FFmpeg:', shutil.which('ffmpeg'))
if not shutil.which('ffmpeg'):
    subprocess.run(['apt-get','update','-qq'], check=True)
    subprocess.run(['apt-get','install','-y','-qq','ffmpeg'], check=True)


## 1. Descargar el código upstream y dependencias mínimas

Se evita `pip install -r requirements.txt`: instala muchos componentes opcionales que no hacen falta para esta prueba y podría consumir demasiado espacio/tiempo.

In [ ]:
REPO_DIR = '/kaggle/working/VideoAgent'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone','--depth','1','https://github.com/HKUDS/VideoAgent.git',REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q',
    'transformers==4.40.1','accelerate==0.30.1','safetensors',
    'tenacity','openai==1.60.1','pydantic==2.9.2','pyyaml','regex','huggingface_hub'], check=True)
print('Código y dependencias mínimas preparados.')


## 2. Leer la clave desde Kaggle Secrets y probar Gemini

La clave no se imprime ni se guarda en Git. En Kaggle, crear un secreto llamado exactamente `GEMINI_API_KEY` y habilitar su acceso para este notebook.

In [ ]:
from kaggle_secrets import UserSecretsClient
from openai import OpenAI
secrets = UserSecretsClient()
GEMINI_API_KEY = secrets.get_secret('GEMINI_API_KEY')
GEMINI_MODEL = 'gemini-2.5-flash'  # cambiar solo si Google indica que este modelo ya no está disponible
client = OpenAI(api_key=GEMINI_API_KEY, base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
check = client.chat.completions.create(model=GEMINI_MODEL, messages=[{'role':'user','content':'Respondé únicamente: CONEXIÓN CORRECTA'}], max_tokens=20)
print(check.choices[0].message.content)


## 3. Configurar el módulo de VideoAgent para usar Gemini

El componente Video QA llama a la función `gpt()` del proyecto. Para esta prueba se enruta esa función al endpoint compatible de Gemini. No se necesitan claves de OpenAI ni Anthropic para ejecutar directamente este componente.

In [ ]:
import yaml
cfg = {'llm': {
    'gpt_api_key': GEMINI_API_KEY,
    'gpt_base_url': 'https://generativelanguage.googleapis.com/v1beta/openai/',
    'gemini_api_key': GEMINI_API_KEY,
    'gemini_base_url': 'https://generativelanguage.googleapis.com/v1beta/openai/',
    'claude_api_key': GEMINI_API_KEY,
    'claude_base_url': 'https://generativelanguage.googleapis.com/v1beta/openai/',
    'deepseek_api_key': '', 'deepseek_base_url': ''
}}
os.makedirs('environment/config', exist_ok=True)
with open('environment/config/config.yml','w',encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True)
llm_path = 'environment/config/llm.py'
llm_text = open(llm_path,encoding='utf-8').read()
llm_text = llm_text.replace('model="gpt-4o"','model="gemini-2.5-flash"')
llm_text = llm_text.replace('model="claude-3-7-sonnet-20250219"','model="gemini-2.5-flash"')
open(llm_path,'w',encoding='utf-8').write(llm_text)
print('Configuración temporal escrita en el entorno de Kaggle.')


## 4. Descargar Whisper large-v3-turbo

Este modelo ocupa bastante espacio. Si la sesión no tiene espacio suficiente, detenerse y no continuar con la descarga.

In [ ]:
from huggingface_hub import snapshot_download
MODEL_DIR = os.path.join(REPO_DIR,'tools','whisper-large-v3-turbo')
os.makedirs(MODEL_DIR, exist_ok=True)
free_gb = shutil.disk_usage('/kaggle/working').free / 1024**3
if free_gb < 5:
    raise RuntimeError(f'Espacio insuficiente ({free_gb:.1f} GB libres). No se descargó Whisper.')
snapshot_download(repo_id='openai/whisper-large-v3-turbo', local_dir=MODEL_DIR)
print('Whisper descargado.')


## 5. Seleccionar un video privado y ejecutar VideoAgent

En el panel **Add Input** de Kaggle, adjuntar un dataset privado que contenga el clip de prueba. Copiar la ruta real del archivo en `VIDEO_PATH`. Para la primera prueba usar un video breve, con audio claro y en español.

In [ ]:
VIDEO_PATH = '/kaggle/input/REEMPLAZAR-DATASET/clip-prueba.mp4'  # completar con la ruta real
if not os.path.isfile(VIDEO_PATH):
    raise FileNotFoundError('No encuentro el video. Adjuntá el dataset privado en Add Input y corregí VIDEO_PATH.')
from environment.roles.vid_qa.content_loader import VideoContentQA
agent = VideoContentQA()
transcript = agent._transcribe_video(VIDEO_PATH)
print(transcript[:12000])


## 6. Preguntar sobre el video usando el agente

La respuesta debe basarse únicamente en la transcripción. Comprobar nombres, cifras y citas contra el video original antes de reutilizar cualquier resultado.

In [ ]:
pregunta = 'Hacé un resumen periodístico fiel del video en español. Identificá los temas, protagonistas, datos y declaraciones principales. Si algo no se entiende en la transcripción, indicálo.'
respuesta = agent._qa_agent(pregunta, f'=== VIDEO: {os.path.basename(VIDEO_PATH)} ===\n{transcript}')
print(respuesta)


## Criterios para decidir si seguimos

- La transcripción en español debe ser suficientemente fiel.
- Gemini debe resumir sin inventar datos y reconocer lo que no aparece en el audio.
- Medir tiempo de descarga, transcripción y respuesta; revisar memoria y espacio usados.
- Esta prueba **no genera un MP4 final ni publica nada**. Si funciona, la siguiente etapa será evaluar el módulo de guion/resumen y el ensamblado separado con FFmpeg.
- No activar el flujo completo `News` de VideoAgent sin reemplazar o excluir ImageBind: el modelo incluido declara licencia CC BY-NC-SA 4.0 y uso solo de investigación/no comercial. El código principal tiene licencia MIT, pero eso no elimina las licencias de los modelos y componentes de terceros.

Kaggle documenta sesiones de notebook con hasta 12 horas, 20 GB de espacio de trabajo y GPU P100 gratuita sujeta a cola/disponibilidad. Esto sirve para pruebas puntuales, no para un servidor permanente.